# 00 Part 4 — Verify Prediction Files

**Purpose:** Confirm all 24 prediction zip files were created correctly before
moving to dataset creation.

**Checks:**
1. All 4 prediction folders exist
2. Each folder has exactly 6 zip files (24 total)
3. Each zip loads without error
4. Embedding and prediction shapes match expected row counts
5. ASIN/date keys align with main_train_keys.csv and main_val_keys.csv

**No GPU needed — CPU runtime is fine.**

## ① Setup

In [ ]:
# Dependencies: pyarrow, pandas (install locally)
print('Local mode')

## ② Config

In [ ]:
import os
import pandas as pd
import numpy as np
import zipfile
import yaml
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

ROOT     = str(PROJECT_ROOT) + '/'
DATA_DIR = ROOT + 'data/'
PRED_DIR = DATA_DIR + 'predictions/'
KEY_DIR  = ROOT + 'code/'
SPLIT_DIR = DATA_DIR + 'amzn_shoes_monthly_diffs_ffill_fixed_splits/'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')

EXPECTED_FOLDERS = [
    'txtimg/lag1',
    'txtimg/time_indipendent',
    'txt/lag1',
    'txt/time_indipendent',
]

print('✅ Config ready')
print(f'Predictions dir: {PRED_DIR}')

## ③ Check All 4 Folders Exist and Have 6 Files Each

In [ ]:
print('=' * 65)
print('FOLDER AND FILE COUNT CHECK')
print('=' * 65)

all_zips = {}
all_ok = True

for folder in EXPECTED_FOLDERS:
    path = PRED_DIR + folder + '/'
    exists = os.path.isdir(path)
    if not exists:
        print(f'  ❌ MISSING folder: {path}')
        all_ok = False
        continue

    zips = sorted([f for f in os.listdir(path) if f.endswith('.zip')])
    count = len(zips)
    status = '✅' if count == 6 else f'❌ expected 6, got {count}'
    print(f'  {status}  {folder}/')
    for z in zips:
        print(f'       {z}')
    all_zips[folder] = [path + z for z in zips]
    print()

total = sum(len(v) for v in all_zips.values())
print(f'Total zip files found: {total} / 24  {"✅" if total == 24 else "❌"}')
print()
if all_ok and total == 24:
    print('✅ All folders and files present — proceeding to content check.')
else:
    print('❌ Some files missing — check which Part 3 notebook needs to be re-run.')

## ④ Load Each Zip and Check Shapes

In [ ]:
print('=' * 65)
print('CONTENT CHECK — LOADING EACH ZIP')
print('=' * 65)
print()

# Load key files to get expected row counts
df_train_keys = pd.read_csv(KEY_DIR + 'main_train_keys.csv')
df_val_keys   = pd.read_csv(KEY_DIR + 'main_val_keys.csv')

print(f'Expected train rows : {len(df_train_keys):,}')
print(f'Expected val rows   : {len(df_val_keys):,}')
print()

load_errors = []

for folder, zip_paths in all_zips.items():
    print(f'--- {folder} ---')
    for zpath in zip_paths:
        fname = os.path.basename(zpath)
        try:
            with zipfile.ZipFile(zpath, 'r') as z:
                csv_files = [f for f in z.namelist() if f.endswith('.csv')]
                if not csv_files:
                    print(f'  ❌ {fname[:60]} — no CSV inside zip')
                    load_errors.append(fname)
                    continue
                df = pd.read_csv(z.open(csv_files[0]))
                rows = len(df)
                cols = len(df.columns)
                # Determine expected rows
                is_train = 'train' in fname.lower()
                is_val   = fname.lower().startswith('val')
                is_diff  = 'diff' in fname
                expected = len(df_train_keys) if is_train else len(df_val_keys) if is_val else '?'
                ok = '✅' if isinstance(expected, str) or abs(rows - expected) < 500 else '⚠️ '
                print(f'  {ok} {fname[:65]}')
                print(f'       rows={rows:,}  cols={cols}')
        except Exception as e:
            print(f'  ❌ {fname[:60]} — ERROR: {e}')
            load_errors.append(fname)
    print()

if not load_errors:
    print('✅ All zip files loaded successfully.')
else:
    print(f'❌ {len(load_errors)} file(s) had errors: {load_errors}')

## ⑤ Check ASIN/Date Key Alignment

In [ ]:
print('=' * 65)
print('KEY ALIGNMENT CHECK')
print('=' * 65)
print()

train_keys = set(zip(df_train_keys['ASIN'], df_train_keys['date'].astype(str)))
val_keys   = set(zip(df_val_keys['ASIN'],   df_val_keys['date'].astype(str)))

# Sample one zip from each folder to check alignment
for folder in EXPECTED_FOLDERS:
    path = PRED_DIR + folder + '/'
    zips = sorted([f for f in os.listdir(path) if f.endswith('.zip')])
    if not zips:
        continue

    # Pick the train level emb256 file
    sample_zip = next((z for z in zips if 'train' in z and 'dim=256' in z), zips[0])
    zpath = path + sample_zip

    try:
        with zipfile.ZipFile(zpath, 'r') as z:
            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
            df = pd.read_csv(z.open(csv_files[0]))

        if 'ASIN' in df.columns and 'date' in df.columns:
            zip_keys  = set(zip(df['ASIN'], df['date'].astype(str)))
            overlap   = len(zip_keys & train_keys)
            pct       = overlap / len(train_keys) * 100 if train_keys else 0
            ok = '✅' if pct > 95 else '⚠️ '
            print(f'  {ok} {folder}/train: {overlap:,} / {len(train_keys):,} keys match ({pct:.1f}%)')
        else:
            print(f'  ⚠️  {folder}: ASIN/date columns not found in {sample_zip}')
    except Exception as e:
        print(f'  ❌ {folder}: {e}')

print()

## ⑥ Final Verdict

In [ ]:
print('=' * 65)
print('FINAL VERIFICATION SUMMARY')
print('=' * 65)

total_found = sum(len(v) for v in all_zips.values())

print(f'''
Folders checked : {len(EXPECTED_FOLDERS)}
Zip files found : {total_found} / 24
Load errors     : {len(load_errors)}
''')

if total_found == 24 and len(load_errors) == 0:
    print('✅ ALL CHECKS PASSED')
    print()
    print('Next steps:')
    print('  1. Run 00_part5_create_image_parquet.ipynb (CPU, ~15 min)')
    print('  2. Run 01_1_create_dataset_txt_img.ipynb   (local machine)')
    print('  3. Run 01_2_create_dataset_txt.ipynb       (local machine)')
else:
    print('❌ SOME CHECKS FAILED — review output above')
    if total_found < 24:
        missing = 24 - total_found
        print(f'   {missing} zip file(s) missing — re-run the corresponding Part 3 notebook')
    if load_errors:
        print(f'   {len(load_errors)} file(s) could not be loaded — may be corrupted')